In [2]:
import os, shutil, random

TRAIN_RATIO = 0.85
images_src = "dataset_1/images"
labels_src = "dataset_1/labels"

images = [f for f in os.listdir(images_src) if f.endswith(".png")]
random.seed(42)
random.shuffle(images)

for folder in ["data/images/train", "data/images/val", "data/labels/train", "data/labels/val"]:
    os.makedirs(folder, exist_ok=True)

for i, img in enumerate(images):
    label = img.replace(".png", ".txt")
    dest = "train" if i < int(len(images) * TRAIN_RATIO) else "val"

    shutil.copy(os.path.join(images_src, img), os.path.join("data/images", dest, img))
    label_path = os.path.join(labels_src, label)
    if os.path.exists(label_path):
        shutil.copy(label_path, os.path.join("data/labels", dest, label))

print(f"Done — train: {int(len(images) * TRAIN_RATIO)}, val: {len(images) - int(len(images) * TRAIN_RATIO)}")

Done — train: 363, val: 65


In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.6.0+cu124
True


In [4]:
from ultralytics import YOLO
from clearml import Task

task = Task.init(project_name="Main", task_name="YOLO Training")

model = YOLO("yolo11n.pt")
model.train(
    data="dataset_1/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    project="runs",
    name="fatigue-v1",
    device=0
)

task.close()

ClearML Task: overwriting (reusing) task id=6aa6067547144d409d3985651d30ebe6
ClearML results page: https://app.clear.ml/projects/bab4f98ba6d147688fafe06e98131cbe/experiments/6aa6067547144d409d3985651d30ebe6/output/log
Ultralytics 8.4.25  Python-3.13.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask

KeyboardInterrupt: 

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO("best.pt")
cap = cv2.VideoCapture(0)  # 0 = default webcam

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)
    annotated = results[0].plot()  # draws boxes + labels on frame

    cv2.imshow("FatigueSense - Live Detection", annotated)

    if cv2.waitKey(1) & 0xFF == ord("q"):  # press Q to quit
        break

cap.release()
cv2.destroyAllWindows()


0: 480x640 2 Eyess, 118.8ms
Speed: 3.6ms preprocess, 118.8ms inference, 3.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 Eyess, 39.9ms
Speed: 4.0ms preprocess, 39.9ms inference, 4.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Mouth, 2 Eyess, 39.6ms
Speed: 5.2ms preprocess, 39.6ms inference, 4.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 Eyess, 35.2ms
Speed: 3.3ms preprocess, 35.2ms inference, 5.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Mouth, 2 Eyess, 34.9ms
Speed: 3.2ms preprocess, 34.9ms inference, 4.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Mouth, 2 Eyess, 36.3ms
Speed: 3.0ms preprocess, 36.3ms inference, 6.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 Eyess, 33.9ms
Speed: 3.3ms preprocess, 33.9ms inference, 4.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 Eyess, 217.3ms
Speed: 3.6ms preprocess, 217.3ms inference, 16.4ms postprocess per 

KeyboardInterrupt: 

: 